# ALADIN preprocessing

Reads the raw ALADIN GRIB forecasts from `data/ALADIN_COMPLETE/<year>/<month>/ashz_*.grb`, regrids them to a regular lat/lon grid with xESMF and writes `data/ALADIN/aladin_merged_ds.nc`, the file `TOK_dataset.py` reads.

This notebook was run interactively, not top to bottom. The extraction and regridding cells were re-run once per variable group (precipitation; `t2m` and `r`; `u10` and `v10`; temperature at 500 hPa), with the extractor and output file changed by hand each time. The merge cell then combines the four regridded files.

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cfgrib
import pandas as pd
import os

os.makedirs("data/ALADIN", exist_ok=True)

In [ ]:
import xarray as xr
import cfgrib
import os
import glob
from tqdm import tqdm
from multiprocess import Pool
from concurrent.futures import ThreadPoolExecutor




# Define the path pattern for all GRIB files
base_dir = "data/ALADIN_COMPLETE"
# Find all GRIB files at once (recursively search through all years and months)
grib_files = sorted(glob.glob(os.path.join(base_dir, "*", "*", "ashz_*.grb")))
print(grib_files)

# Function to open and extract 'tp' field from a single file
def extract_tp(filepath):
    import cfgrib
    try:
        try:
            ds = cfgrib.open_datasets(filepath, filter_by_keys={'typeOfLevel': 'surface'},backend_kwargs={
                'filter_by_keys': {'typeOfLevel': 'surface',}
            }, engine='cfgrib', decode_timedelta = True)[-2][['tp']].load()
            return ds  # Select only total precipitation
        except Exception as e:
            print(f"Error processing {filepath}: {e}")
            return e
    except Exception as e:
        print(f"Error opening {filepath}: {e}", flush=True)
        return e


#results = [extract_tp(f) for f in tqdm(grib_files)]


with Pool(16) as executor: 
    print(grib_files)
    results = list(tqdm(executor.imap_unordered(extract_tp, grib_files,chunksize=20), total=len(grib_files), leave=False))


#final_ds = xr.concat(results, dim="time")


In [ ]:
len(results)

In [ ]:
final_ds = xr.concat(results, dim="time")
# Keep only first 24 steps out of 72
final_ds = final_ds.isel(step=slice(0,24))


In [ ]:
import numpy as np
# results are indexed on curvilinear grid. We would like to resample them to regular grid
# this is our targeted regridding
aladin_raw_latitude = np.array([45.202, 45.242,
        45.282, 45.322, 45.362, 45.402, 45.442, 45.482, 45.522, 45.562,
        45.602, 45.642, 45.682, 45.722, 45.762, 45.802, 45.842, 45.882,
        45.922, 45.962, 46.002, 46.042, 46.082, 46.122, 46.162, 46.202,
        46.242, 46.282, 46.322, 46.362, 46.402, 46.442, 46.482, 46.522,
        46.562, 46.602, 46.642, 46.682, 46.722, 46.762, 46.802, 46.842,
        46.882, 46.922, 46.962, 47.002, 47.042])
aladin_raw_longitude = np.array([13.05 , 13.107, 13.164, 13.221, 13.278, 13.335, 13.392,
        13.449, 13.506, 13.563, 13.62 , 13.677, 13.734, 13.791, 13.848,
        13.905, 13.962, 14.019, 14.076, 14.133, 14.19 , 14.247, 14.304,
        14.361, 14.418, 14.475, 14.532, 14.589, 14.646, 14.703, 14.76 ,
        14.817, 14.874, 14.931, 14.988, 15.045, 15.102, 15.159, 15.216,
        15.273, 15.33 , 15.387, 15.444, 15.501, 15.558, 15.615, 15.672,
        15.729, 15.786, 15.843, 15.9  , 15.957, 16.014, 16.071, 16.128,
        16.185, 16.242, 16.299, 16.356, 16.413, 16.47 , 16.527, 16.584,
        16.641, 16.698, 16.755, 16.812, 16.869, 16.926, 16.983, 17.04])

lon_new_2d, lat_new_2d = np.meshgrid(aladin_raw_longitude, aladin_raw_latitude)

import xesmf as xe
# Create regridder object
regridder = xe.Regridder(final_ds, {'lon': aladin_raw_longitude, 'lat': aladin_raw_latitude}, method='bilinear')
# Apply regridding to the dataset
ds_regular = regridder(final_ds)


In [ ]:
#ds_regular.tp[0][0].plot()
#final_ds.tp[0][0].plot()
print(f'size is {ds_regular.tp.nbytes/final_ds.tp.nbytes*100:0.2f}% of original, from {final_ds.tp.nbytes/1e9} gb to {ds_regular.tp.nbytes/1e9}')

# drop any row where tp is nan, sort by time
ds_regular = ds_regular.dropna(dim='time', how='any', subset=['tp']).sortby('time')

In [ ]:
ds_regular

In [ ]:
# Save the regridded dataset
#ds_regular.to_netcdf('data/ALADIN/aladin_tp_regridded.nc')


In [ ]:
# Convert accumulative tp to hourly intervals
# tp is accumulative, so we need to compute differences between consecutive steps
ds_regular_hourly = ds_regular.copy()

# Create a new array for hourly tp values
tp_hourly = ds_regular.tp.copy()

# Use a for loop to compute differences between consecutive steps
for step_idx in range(1, tp_hourly.shape[1]):
    tp_hourly[:, step_idx, :, :] = ds_regular.tp[:, step_idx, :, :] - ds_regular.tp[:, step_idx-1, :, :]

# Set negative values to 0
tp_hourly = tp_hourly.where(tp_hourly >= 0, 0)

# Update the dataset with hourly tp values
ds_regular_hourly['tp'] = tp_hourly

ds_regular_hourly.to_netcdf('data/ALADIN/aladin_tp_regridded_HOURLY_ACC.nc')

In [ ]:
ds2 = cfgrib.open_datasets(grib_files[12],backend_kwargs={
            'indexpath': ''
        }, engine='cfgrib', decode_timedelta = True)

In [ ]:
print(len(ds2))
for x in ds2:
    print(x.data_vars)

In [ ]:
ds = cfgrib.open_datasets(grib_files[-12],backend_kwargs={
            'indexpath': ''
        }, engine='cfgrib', decode_timedelta = True)

In [ ]:
ds[3][['t2m','r']]

In [ ]:
print(len(ds))
for x in ds:
    print(x.data_vars)

In [ ]:
ds[2] # za temperaturo in vlago
ds[3] # veter
(ds[5].sel(isobaricInhPa=500).t.to_numpy() - ds[2].t2m.to_numpy())

In [ ]:
import xarray as xr
import cfgrib
import os
import glob
from tqdm import tqdm
from multiprocess import Pool
from concurrent.futures import ThreadPoolExecutor
# Define the path pattern for all GRIB files
base_dir = "data/ALADIN_COMPLETE"
# Find all GRIB files at once (recursively search through all years and months)
grib_files = sorted(glob.glob(os.path.join(base_dir, "*", "*", "ashz_*.grb")))
print(grib_files)
# Function to open and extract 'tp' field from a single file
def extract_t2m_r(filepath):
    import cfgrib
    import os
    try:
        try:
            # Extract year from filepath to determine which dataset index to use
            year = int(os.path.basename(filepath).split('_')[1][:4])
            dataset_index = 3 if year >= 2019 else 2
            ds = cfgrib.open_datasets(filepath,backend_kwargs={
            }, engine='cfgrib', decode_timedelta = True)[dataset_index][['t2m','r']].isel(step=slice(0,24)).load()
            return ds  # Select only total precipitation
        except Exception as e:
            print(f"Error processing {filepath}: {e}")
            return e
    except Exception as e:
        print(f"Error opening {filepath}: {e}", flush=True)
        return e
    
def extract_wind(filepath):
    import cfgrib
    import os
    try:
        try:
            year = int(os.path.basename(filepath).split('_')[1][:4])
            dataset_index = 0 if year >= 2019 else 3
            ds = cfgrib.open_datasets(filepath,backend_kwargs={
            }, engine='cfgrib', decode_timedelta = True)[dataset_index][['u10','v10']].isel(step=slice(0,24)).load()
            return ds  # Select only total precipitation
        except Exception as e:
            print(f"Error processing {filepath}: {e}")
            return e
    except Exception as e:
        print(f"Error opening {filepath}: {e}", flush=True)
        return e
    
def extract_t_500hPa(filepath):
    import cfgrib
    import os
    try:
        try:
            ds = cfgrib.open_datasets(filepath,backend_kwargs={
            }, engine='cfgrib', decode_timedelta = True)[5].sel(isobaricInhPa=500)[['t']].isel(step=slice(0,24)).load()
            return ds  # Select only total precipitation
        except Exception as e:
            print(f"Error processing {filepath}: {e}")
            return e
    except Exception as e:
        print(f"Error opening {filepath}: {e}", flush=True)
        return e
    


In [ ]:
with Pool(16) as executor: 
    print(grib_files)
    #results_t2m_r = list(tqdm(executor.imap_unordered(extract_t2m_r, grib_files,chunksize=20), total=len(grib_files), leave=False))
    #results_wind = list(tqdm(executor.imap_unordered(extract_wind, grib_files,chunksize=20), total=len(grib_files), leave=False))
    results_t_500hPa = list(tqdm(executor.imap_unordered(extract_t_500hPa, grib_files,chunksize=20), total=len(grib_files), leave=False))

In [ ]:
results = results_t_500hPa

final_ds = xr.concat(results, dim="time")
# Keep only first 24 steps out of 72
final_ds = final_ds.isel(step=slice(0,24))

import numpy as np
# results are indexed on curvilinear grid. We would like to resample them to regular grid
# this is our targeted regridding
aladin_raw_latitude = np.array([45.202, 45.242,
        45.282, 45.322, 45.362, 45.402, 45.442, 45.482, 45.522, 45.562,
        45.602, 45.642, 45.682, 45.722, 45.762, 45.802, 45.842, 45.882,
        45.922, 45.962, 46.002, 46.042, 46.082, 46.122, 46.162, 46.202,
        46.242, 46.282, 46.322, 46.362, 46.402, 46.442, 46.482, 46.522,
        46.562, 46.602, 46.642, 46.682, 46.722, 46.762, 46.802, 46.842,
        46.882, 46.922, 46.962, 47.002, 47.042])
aladin_raw_longitude = np.array([13.05 , 13.107, 13.164, 13.221, 13.278, 13.335, 13.392,
        13.449, 13.506, 13.563, 13.62 , 13.677, 13.734, 13.791, 13.848,
        13.905, 13.962, 14.019, 14.076, 14.133, 14.19 , 14.247, 14.304,
        14.361, 14.418, 14.475, 14.532, 14.589, 14.646, 14.703, 14.76 ,
        14.817, 14.874, 14.931, 14.988, 15.045, 15.102, 15.159, 15.216,
        15.273, 15.33 , 15.387, 15.444, 15.501, 15.558, 15.615, 15.672,
        15.729, 15.786, 15.843, 15.9  , 15.957, 16.014, 16.071, 16.128,
        16.185, 16.242, 16.299, 16.356, 16.413, 16.47 , 16.527, 16.584,
        16.641, 16.698, 16.755, 16.812, 16.869, 16.926, 16.983, 17.04])

lon_new_2d, lat_new_2d = np.meshgrid(aladin_raw_longitude, aladin_raw_latitude)

import xesmf as xe
# Create regridder object
regridder = xe.Regridder(final_ds, {'lon': aladin_raw_longitude, 'lat': aladin_raw_latitude}, method='bilinear')
# Apply regridding to the dataset
ds_regular = regridder(final_ds)


In [ ]:
#ds_regular
ds_regular = ds_regular[['t']]
ds_regular
#ds_regular[['t']].dropna(dim='time', how='any', subset=['t']).sortby('time').to_netcdf('data/ALADIN/aladin_t500hPa_regridded.nc')


In [ ]:
DEMO = xr.open_dataset('data/ALADIN/aladin_tp_regridded_HOURLY_ACC.nc')


In [ ]:
DEMO

In [ ]:
print(f'size is {ds_regular.tp.nbytes/final_ds.tp.nbytes*100:0.2f}% of original, from {final_ds.tp.nbytes/1e9} gb to {ds_regular.tp.nbytes/1e9}')
# drop any row where tp is nan, sort by time
ds_regular = ds_regular.dropna(dim='time', how='any', subset=['tp']).sortby('time')

ds_regular.to_netcdf('data/ALADIN/aladin_t2m_r_regridded.nc')

In [ ]:
#
# concat all data into one dataset
#
tp = xr.open_dataset('data/ALADIN/aladin_tp_regridded_HOURLY_ACC.nc', decode_timedelta=True).load()
t2m_r = xr.open_dataset('data/ALADIN/aladin_t2m_r_regridded.nc', decode_timedelta=True).load()
wind = xr.open_dataset('data/ALADIN/aladin_wind_regridded.nc', decode_timedelta=True).load()
t500hPa = xr.open_dataset('data/ALADIN/aladin_t500hPa_regridded.nc', decode_timedelta=True).load()

In [ ]:
# Merge the first 3 datasets (tp, t2m_r, wind)
t2m_r_fixed = t2m_r.drop_vars('heightAboveGround')
t2m_r_fixed = t2m_r_fixed.assign_coords(step=tp.step, valid_time=tp.valid_time)
t2m_r_fixed['t2m'] = t2m_r_fixed.t2m/330 # normalize temperature, humidity is already normalized

wind_fixed = wind.drop_vars(['heightAboveGround'])
wind_fixed = wind_fixed.assign_coords(step=tp.step, valid_time=tp.valid_time)
wind_fixed['u10'] = wind_fixed.u10/30
wind_fixed['v10'] = wind_fixed.v10/30

t500hPa_fixed = t500hPa.drop_vars(['isobaricInhPa'])
t500hPa_fixed = t500hPa_fixed.assign_coords(step=tp.step, valid_time=tp.valid_time)
t500hPa_fixed['diffT500hPa'] = t2m_r_fixed.t2m - t500hPa_fixed.t/330
t500hPa_fixed = t500hPa_fixed.drop_vars('t')
t500hPa_fixed


merged_ds = xr.merge([tp, t2m_r_fixed, wind_fixed, t500hPa_fixed])


In [ ]:
print(f'Merged dataset size: {merged_ds.nbytes/1e9:.2f} GB')
merged_ds.to_netcdf('data/ALADIN/aladin_merged_ds.nc')

In [ ]:
# Read the merged dataset
merged_ds = xr.open_dataset('data/ALADIN/aladin_merged_ds.nc', decode_timedelta=True)
merged_ds


In [ ]:
import xarray as xr
import matplotlib.pyplot as plt

# Load the dataset
ds = xr.open_dataset('data/ALADIN/aladin_merged_ds.nc')

# Select data for 2023-08-04
date = '2023-08-04'
ds_date = ds.sel(time=date)

In [ ]:
ds.u10[0,21].plot()

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# tp plot
date = '2023-08-05'
ds_date = ds.sel(time=date)
print('tp')
fig = plt.figure(figsize=(8, 6))
ax = plt.axes(projection=ccrs.PlateCarree())
ds_date.tp.isel(lon=slice(60,65),lat=slice(40,45))[12].plot(ax=ax, transform=ccrs.PlateCarree(), add_colorbar=False, add_labels=False, cmap='Blues')
ax.add_feature(cfeature.BORDERS, linewidth=1, edgecolor='black')
ax.add_feature(cfeature.COASTLINE, linewidth=1, edgecolor='black')
plt.show()
fig = plt.figure(figsize=(8, 6))
ax = plt.axes(projection=ccrs.PlateCarree())
ds_date.tp.isel(lon=slice(60,65),lat=slice(40,45))[13].plot(ax=ax, transform=ccrs.PlateCarree(), add_colorbar=False, add_labels=False, cmap='Blues')
ax.add_feature(cfeature.BORDERS, linewidth=1, edgecolor='black')
ax.add_feature(cfeature.COASTLINE, linewidth=1, edgecolor='black')
plt.show()
fig = plt.figure(figsize=(8, 6))
ax = plt.axes(projection=ccrs.PlateCarree())
ds_date.tp.isel(lon=slice(60,65),lat=slice(40,45))[14].plot(ax=ax, transform=ccrs.PlateCarree(), add_colorbar=False, add_labels=False, cmap='Blues')
ax.add_feature(cfeature.BORDERS, linewidth=1, edgecolor='black')
ax.add_feature(cfeature.COASTLINE, linewidth=1, edgecolor='black')
plt.show()
fig = plt.figure(figsize=(8, 6))
ax = plt.axes(projection=ccrs.PlateCarree())
ds_date.tp.isel(lon=slice(60,65),lat=slice(40,45))[15].plot(ax=ax, transform=ccrs.PlateCarree(), add_colorbar=False, add_labels=False, cmap='Blues')
ax.add_feature(cfeature.BORDERS, linewidth=1, edgecolor='black')
ax.add_feature(cfeature.COASTLINE, linewidth=1, edgecolor='black')
plt.show()

# r plot
date = '2023-06-15'
ds_date = ds.sel(time=date)
print('r')
fig = plt.figure(figsize=(8, 6))
ax = plt.axes(projection=ccrs.PlateCarree())
ds_date.r.isel(lon=slice(5,11),lat=slice(3,8)).sum(dim='step').plot(ax=ax, transform=ccrs.PlateCarree(), add_colorbar=False, add_labels=False, cmap='Greens')
ax.add_feature(cfeature.BORDERS, linewidth=1, edgecolor='black')
ax.add_feature(cfeature.COASTLINE, linewidth=1, edgecolor='black')
plt.show()

# u10 plot
date = '2023-03-22'
ds_date = ds.sel(time=date)
print('u10')
fig = plt.figure(figsize=(8, 6))
ax = plt.axes(projection=ccrs.PlateCarree())
ds_date.u10.isel(lon=slice(6,11),lat=slice(3,8)).sum(dim='step').plot(ax=ax, transform=ccrs.PlateCarree(), add_colorbar=False, add_labels=False)
ax.add_feature(cfeature.BORDERS, linewidth=1, edgecolor='black')
ax.add_feature(cfeature.COASTLINE, linewidth=1, edgecolor='black')
plt.show()

# diffT plot
date = '2023-11-10'
ds_date = ds.sel(time=date)
print('diffT')
fig = plt.figure(figsize=(8, 6))
ax = plt.axes(projection=ccrs.PlateCarree())
ds_date.diffT500hPa.isel(lon=slice(6,11),lat=slice(3,8)).sum(dim='step').plot(ax=ax, transform=ccrs.PlateCarree(), add_colorbar=False, add_labels=False, cmap='Purples')
ax.add_feature(cfeature.BORDERS, linewidth=1, edgecolor='black')
ax.add_feature(cfeature.COASTLINE, linewidth=1, edgecolor='black')
plt.show()

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt

# Load the dataset
ds = xr.open_dataset('data/ALADIN/aladin_merged_ds.nc')

# Select data for 2023-08-04
date = '2023-08-04'
ds_date = ds.sel(time=date)

# Create a figure for precipitation
plt.figure(figsize=(10, 8))

# Plot precipitation
ds_date['tp'].sel(lon=slice(14.15, 14.6), lat=slice(46, 46.3)).sum(dim='step').plot(cmap='viridis')
plt.title(f'Precipitation - {date}')
plt.xlabel('Longitude')
plt.ylabel('Latitude')

# Add more ticks to both axes with 0.05 spacing
plt.gca().xaxis.set_major_locator(plt.MultipleLocator(0.05))
plt.gca().yaxis.set_major_locator(plt.MultipleLocator(0.05))

# Rotate x-axis labels for better readability
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
ds_date['tp'].sel(lon=slice(14.15, 14.6), lat=slice(46, 46.3)).sum(dim='step')

In [ ]:
t2m_r.drop_vars('heightAboveGround')